# *Раздел I. Простая линейная регрессия*

Импортируем библиотеки

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Обработка ошибок
import warnings
warnings.filterwarnings("ignore")

Установим стиль для графиков

In [ ]:
plt.style.use('ggplot')

Загрузим набор данных и отобразим первые 5 строк

In [1]:
df = pd.read_csv('data/weight-height.csv')
df.head()

NameError: name 'pd' is not defined

Выведем информацию о структуре датасета

In [ ]:
# Размер
df.shape

In [ ]:
# Информация о типах данных
df.dtypes

In [ ]:
# Подробная информация о датасете
df.info()

Выше видно, что нулевых значений нет. Но можем это дополнительно проверить

In [ ]:
df.isna().sum()

Пропущенных значений не обнаружено. Можем двигаться дальше

Создадим гистограммы для роста и веса

In [ ]:
# Гистограмма для роста
df.Height.plot(kind='hist',
              color='orange', edgecolor='black', alpha=0.6,
              figsize=(8, 5))
plt.title('Распределение роста', size=18)
plt.xlabel('Рост (дюймы)', size=14)
plt.ylabel('Частота', size=14)
plt.show()

In [ ]:
# Гистограмма для веса
df.Weight.plot(kind='hist',
              color='purple', edgecolor='black', alpha=0.6,
              figsize=(8, 5))
plt.title('Распределение веса', size=18)
plt.xlabel('Вес (фунты)', size=14)
plt.ylabel('Частота', size=14)
plt.show()

Гистограммы имеют нормальное распределение. Также, можно выявить зависимость, что, чем выше человек, тем он тяжелее.

Нормальное распределение говорит нам о том, что наименьшие и наибольшие значения роста и веса встречаются крайне редко.

Разделим гистограммы по гендеру

In [ ]:
# Гистограмма для роста мужчин и женщин
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,6))
df[df['Gender'] == 'Male'].Height.plot(kind='hist',
                                       color='blue',
                                       edgecolor='black',
                                       alpha=0.5,
                                       ax=ax1)

df[df['Gender'] == 'Female'].Height.plot(kind='hist',
                                         color='red',
                                         edgecolor='black',
                                         alpha=0.5,
                                         ax=ax2)
plt.suptitle('Распределение роста', size=18)
ax1.set_title('Мужчины', size=16)
ax1.set_xlabel('Рост (дюймы)', size=14)
ax1.set_ylabel('Частота', size=14)
ax2.set_title('Женщины', size=16)
ax2.set_xlabel('Рост (дюймы)', size=14)
ax2.set_ylabel('Частота', size=14)
plt.show()

In [ ]:
# Гистограмма для веса мужчин и женщин
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,6))
df[df['Gender'] == 'Male'].Weight.plot(kind='hist',
                                       color='blue',
                                       edgecolor='black',
                                       alpha=0.5,
                                       ax=ax1)

df[df['Gender'] == 'Female'].Weight.plot(kind='hist',
                                         color='red',
                                         edgecolor='black',
                                         alpha=0.5,
                                         ax=ax2)
plt.suptitle('Распределение веса', size=18)
ax1.set_title('Мужчины', size=16)
ax1.set_xlabel('Вес (фунты)', size=14)
ax1.set_ylabel('Частота', size=14)
ax2.set_title('Женщины', size=16)
ax2.set_xlabel('Вес (фунты)', size=14)
ax2.set_ylabel('Частота', size=14)
plt.show()

Графики показывают, что рост и вес имеют нормальное распределение для мужчин и женщин. 

Выделим отдельно статистику по гендерам и выведем её общим фреймом

In [ ]:
# Статистика по мужчинам
statistics_male = df[df['Gender'] == 'Male'].describe()
statistics_male.rename(columns=lambda x: x + '_male', inplace=True)

# Статистика по женщинам
statistics_female = df[df['Gender'] == 'Female'].describe()
statistics_female.rename(columns=lambda x: x + '_female', inplace=True)

# Общая статтистика
statistics = pd.concat([statistics_male, statistics_female], axis=1)
statistics

Значения показателей мы можем отобразить в форме точек с помощью точечного графика.
Но если вывести все 10 000 точек на одном полотне, то произвойдёт перекрытие, и мы не сможем ничего разглядеть.

In [ ]:
fig_, (ax_1, ax_2) = plt.subplots(1, 2, figsize=(12,6))
ax_1 = df[df['Gender'] == 'Male'].plot(kind='scatter',
                                       x='Height',
                                       y='Weight',
                                       color='blue',
                                       alpha=0.5,
                                       ax=ax_1)

ax_2 = df[df['Gender'] == 'Female'].plot(kind='scatter',
                                       x='Height',
                                       y='Weight',
                                       color='red',
                                       alpha=0.5,
                                       ax=ax_2)

ax_1.set_title('Мужчины', size=16)
ax_1.set_xlabel('Рост (дюймы)', size=14)
ax_1.set_ylabel('Вес (фунты)', size=14)
ax_1.legend(['Male'])

ax_2.set_title('Женщины', size=16)
ax_2.set_xlabel('Рост (дюймы)', size=14)
ax_2.set_ylabel('Вес (фунты)', size=14)
ax_2.legend(['Female'])
plt.suptitle('Зависимость между ростом и весом', size=18)

plt.show()

Полученные графики всё равно имеют довольно сильное перекрытие, поэтому мы можем взять отдельную выборку из 500 случайных записей

In [ ]:
fig_, (ax_1, ax_2) = plt.subplots(1, 2, figsize=(12,6))
ax_1 = df[df['Gender'] == 'Male'].sample(500).plot(kind='scatter',
                                       x='Height',
                                       y='Weight',
                                       color='blue',
                                       alpha=0.5,
                                       ax=ax_1)

ax_2 = df[df['Gender'] == 'Female'].sample(500).plot(kind='scatter',
                                       x='Height',
                                       y='Weight',
                                       color='red',
                                       alpha=0.5,
                                       ax=ax_2)

ax_1.set_title('Мужчины', size=16)
ax_1.set_xlabel('Рост (дюймы)', size=14)
ax_1.set_ylabel('Вес (фунты)', size=14)
ax_1.legend(['Male'])

ax_2.set_title('Женщины', size=16)
ax_2.set_xlabel('Рост (дюймы)', size=14)
ax_2.set_ylabel('Вес (фунты)', size=14)
ax_2.legend(['Female'])
plt.suptitle('Зависимость между ростом и весом для 500 мужчин и женщин', size=18)

plt.show()

Как видно на графиках, вес мужчин и женщин возрастает с увеличением роста. 

Вычислим простую линейную регрессию, которая описывает отношения между двумя переменными с помощью линейного уравнения, с использованием библиотеки NumPy


Нам необходимо построить такую линию, которая лучше всего соответствует исходным данным и при этом минимизирует сумму квадратичных ошибок 

In [ ]:
# Наилучшее приблежение полином
df_males = df[df['Gender'] == 'Male']
df_females = df[df['Gender'] == 'Female']

# Полином для мужчин
male_fit = np.polyfit(x=df_males.Height, y=df_males.Weight, deg=1)

# Полином для женщим
female_fit = np.polyfit(x=df_females.Height, y=df_females.Weight, deg=1)

# Возвращаются коэффициенты (b, a)
print(f'Мужчины: a = {male_fit[1]}, b = {male_fit[0]}')
print(f'Женщины: a = {female_fit[1]}, b = {female_fit[0]}')

Здесь параметр deg отвечает за степень полинома (многочлена).

Так как мы строим линейную регрессию, то у нас должен быть многочлен первой степени.

Отобразим наш точечный график вместе с полученной линейной регрессией

In [ ]:
ax1 = df_males.plot(kind='scatter', x='Height', y='Weight', color='blue', alpha=0.5,
                    figsize=(10, 7))
df_females.plot(kind='scatter', x='Height', y='Weight', color='magenta', alpha=0.5,
                    figsize=(10, 7), ax=ax1)

# Линии регрессии
plt.plot(df_males.Height, male_fit[0]*df_males.Height+male_fit[1], color='darkblue', linewidth=2)
plt.plot(df_females.Height, female_fit[0]*df_females.Height+female_fit[1], color='deeppink', linewidth=2)

# Формула регрессии
plt.text(65, 230, 'y={:.2f}+{:.2f}*x'.format(male_fit[1], male_fit[0]), color='darkblue', size=12)
plt.text(70, 130, 'y={:.2f}+{:.2f}*x'.format(female_fit[1], female_fit[0]), color='deeppink', size=12)

# Легенда, название, оси
plt.legend(labels=['Мужчины', 'Женщины', 'Линия Регрессии Мужчин', 'Линия Регрессии Женщин'])
plt.title('Отношение между Ростом и Весом', size=24)
plt.xlabel('Рост (дюймы)', size=18)
plt.ylabel('Вес (фунты)', size=18)

plt.show()

Построим график с регрессией, используя библиотеку Seaborn, основанную на Matplotlib

Чтобы отключить доверительный интервал - полупрозрачную область, прописал атрибут ci=None

In [ ]:
# Выборка
df_males_sample = df_males.sample(300)
df_females_sample = df_females.sample(300)

# График регрессии
fig = plt.figure(figsize=(10,7))
sns.regplot(x=df_males_sample.Height,
            y=df_males_sample.Weight,
            color='blue', marker='+', ci=None)

sns.regplot(x=df_females_sample.Height,
            y=df_females_sample.Weight,
            color='magenta', marker='+', ci=None)

plt.legend(labels=['Мужчины', 'Линия Регрессии Мужчин', 'Женщины', 'Линия Регрессии Женщин', 'girl'])
plt.title('Отношение между Ростом и Весом', size=24)
plt.xlabel('Рост (дюймы)', size=18)
plt.ylabel('Вес (фунты)', size=18)

plt.show()

Теперь подгоним модель с использованием специализированной библиотеки.

Для это из библиотеки sklear импортируем класс Линейной регрессии.

In [ ]:
from sklearn.linear_model import LinearRegression

lr_males = LinearRegression()
lr_males.fit(df_males[['Height']], df_males[['Weight']])

print(f'Мужчины, a: {lr_males.intercept_[0]}, b:{lr_males.coef_[0]}')

lr_females = LinearRegression()
lr_females.fit(df_females[['Height']], df_females[['Weight']])

print(f'Женщины, a: {lr_females.intercept_[0]}, b:{lr_females.coef_[0]}')

После подгона модели можно построить функцию для прогнозирования веса по росту.

У нас есть по две модели для каждого пола: есть линейная регрессия, которая подгонялась через NumPy, а есть, которая подгонялась с помощью Sklearn

In [ ]:
# Прогнозирование с помощью NumPy
print(f'Прогнозирование для мужчины с помощью NumPy: {np.polyval(male_fit, 60)}')
print(f'Прогнозирование для женщины с помощью NumPy: {np.polyval(female_fit, 60)}')
# Прогнозирование с помощью Sklearn
print(f'Прогнозирование для мужчины с помощью Sklearn: {lr_males.predict([[60]])[0]}')
print(f'Прогнозирование для женщины с помощью Sklearn: {lr_females.predict([[60]])[0]}')

Также мы можем выявить силу и направление отношений между двумя переменными, которую называют коэффициентом корреляции Пирсона.

Для этого можно воспользоваться как библиотекой Pandas, так и Sklearn

In [ ]:
# Коэффициент корреляции Пирсона. Pandas
# Женщины
df_females_cor = df_females.iloc[:, 1:]
print('Females')
df_females_cor.corr()

In [ ]:
# Коэффициент корреляции Пирсона. Pandas
# Мужчины
df_males_cor = df_males.iloc[:, 1:]
print('Males')
df_males_cor.corr()

In [ ]:
# Коэффициент корреляции Пирсона. Sklearn
pearson_coef_f, p_value_f = stats.pearsonr(df_females.Height, df_females.Weight)
print(f'Коэффициент для женщин: {pearson_coef_f}')

pearson_coef_m, p_value_m = stats.pearsonr(df_males.Height, df_males.Weight)
print(f'Коэффициент для мужчин: {pearson_coef_m}')

Таким образом, коэффициент корреляции с помощью Pandas и с помощью SciPy совпали.

По значениям видно, что между переменными роста и веса очень сильные линейные отношения.

Мы выполнили оценку, можно ли выполнять прогнозирование данных с помощью модели, построенной на основе линейной регрессии,
с помощью выявления линейной зависимости переменных. Но есть и другой способ.

Вторым способом является использование остаточных участков. Они показывают разницу между фактическими и прогнозируемыми значениями.

In [ ]:
# Женщины
df_females_sample = df_females.sample(500)

fig = plt.figure(figsize=(10, 7))
sns.residplot(x=df_females_sample.Height,
              y=df_females_sample.Weight,
              color='magenta')

plt.title('Остаточный график по 500 женщинам')
plt.xlabel('Рост (дюймы)')
plt.ylabel('Вес (фунты)')

plt.show()

По графику видно, что значения распределены вокруг нуля, что означает, что линейная регрессионная модель подходит для прогнозирования данных.

In [ ]:
# Мужчины
df_males_sample = df_males.sample(500)

fig = plt.figure(figsize=(10, 7))
sns.residplot(x=df_males_sample.Height,
              y=df_males_sample.Weight,
              color='blue')

plt.title('Остаточный график по 500 мужчинам')
plt.xlabel('Рост (дюймы)')
plt.ylabel('Вес (фунты)')

plt.show()

Для мужчин ситуация аналогична

# *Раздел II. Множественная линейная регрессия*

## *Цель исследования:*

Определить стоимость на недвижимость в зависимость от её местоположения, числа комнат, типа дома и площадей жилья.

### *Задачи:*
* Выявить зависимость стоимости от параметров;
* Построить модель множественной линейной регрессии;
* Предсказать стоимость некоторой выборки недвижимости;
* Оценить модель несколькими способами.

В файле заменил значения с лишними пробелами с помощью Microsoft Excel

In [ ]:
# Импортирование библиотек

In [ ]:
import pandas as pd
import numpy as np
from sklearn import linear_model
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
# Чтение файла
df = pd.read_excel('nedvig.xlsx')
df

In [ ]:
print(df.head())
print(df.shape)
print(df.dtypes)
print(df.info())
print(df.nunique())
print(df.isna().sum())

В таблице присутствуют 2 столбца с пропущенными значениями. Нужно заменить их на значения, равные математическому ожиданию.

In [ ]:
df[['Жилая', 'Кухня']] = df[['Жилая', 'Кухня']].apply(lambda x: x.fillna(x.mean()))
print(df.isna().sum())

В таблице содержится 1573 строки и 7 столбцов. Данный набор представляет собой зафиксированные параметры квартир, куда вошли и район расположения недвижимости - всего их 23 и представлены в виде категориальных данных; ещё в таком виде представлен тип дома, из числовых данных здесь описаны: число комнат - 5 разных значений, общая площадь, жилая площадь и площадь кухни. В последнем столбце содержится стоимость этой недвижимости в тыс. рублей.

Посмотрим распределение категориальных данных по столбцу с типом домов.

In [ ]:
# Выведем все типы домов и колчи

df['Тип дома'].value_counts()

Построим описательную статистику наиболее численных категорий.

In [ ]:
stat_block = df[df['Тип дома'] == "блочный"].describe()
stat_block.rename(columns=lambda x: x + "_bl", inplace=True)

stat_brick =df[df['Тип дома'] == "кирпичный"].describe()
stat_brick.rename(columns=lambda x: x + "_br", inplace=True)

stat_const =df[df['Тип дома'] == "каркасный"].describe()
stat_const.rename(columns=lambda x: x + "_const", inplace=True)

statistic = pd.concat([stat_block, stat_brick, stat_const], axis=1)
statistic

### Построим графики

In [ ]:
# Распределение стоимости по району
ax1 = df.plot(kind='scatter',
              x='Район', y='Цена, тыс. руб.',
              color = 'orange', alpha=0.6,
              figsize=(15, 7))

plt.xticks(rotation=90)
plt.title("Распределение стоимости недвижимости по районам")
plt.show()

Построим некоторые гистограммы.

In [ ]:
# Распределение общей площади
df['Общая'].plot(kind='hist',
        color='purple', edgecolor='black', alpha=0.6,
        figsize=(8, 6))

plt.title('Распределение общей площади', size=18)
plt.xlabel('Общая площадь (кв. м.)', size=14)
plt.ylabel('Частота', size=14)
plt.show()

In [ ]:
# Распределение жилой площади
df['Жилая'].plot(kind='hist',
        color='pink', edgecolor='black', alpha=0.6,
        figsize=(8, 6))

plt.title('Распределение жилой площади', size=18)
plt.xlabel('Жилая площадь (кв. м.)', size=14)
plt.ylabel('Частота', size=14)
plt.show()

In [ ]:
# Распределение площади кухни
df['Кухня'].plot(kind='hist',
        color='lightblue', edgecolor='black', alpha=0.6,
        figsize=(8, 6))

plt.title('Распределение площади кухни', size=18)
plt.xlabel('Площадь кухни (кв. м.)', size=14)
plt.ylabel('Частота', size=14)
plt.show()

In [ ]:
# Распределение цены
df['Цена, тыс. руб.'].plot(kind='hist',
        color='green', edgecolor='black', alpha=0.6,
        figsize=(10, 6))

plt.title('Распределение цены', size=18)
plt.xlabel('Цена (тыс. руб.)', size=14)
plt.ylabel('Частота', size=14)
plt.show()

In [ ]:
# Распределение стоимости по общей площади
ax1 = df[df['Тип дома'] == 'блочный'].plot(kind='scatter',
              x='Общая', y='Цена, тыс. руб.',
              color = 'orange', alpha=0.6,
              figsize=(10, 7))

df[df['Тип дома'] == 'каркасный'].plot(kind='scatter',
              x='Общая', y='Цена, тыс. руб.',
              color = 'blue', alpha=0.6,
              figsize=(10, 7), ax=ax1)

df[df['Тип дома'] == 'кирпичный'].plot(kind='scatter',
              x='Общая', y='Цена, тыс. руб.',
              color = 'red', alpha=0.6,
              figsize=(10, 7), ax=ax1)

plt.show()

На графике видно, что есть одно значение, которое очень сильно отдалено от остальных точек, поэтому найдем строку и удалим её из датасета.

In [ ]:
print(df[df['Цена, тыс. руб.'] == df['Цена, тыс. руб.'].max()])

In [ ]:
# Удаление
df.drop(index=146, inplace=True)
df.reset_index(drop=True, inplace=True)

# Проверка
print(df[df['Цена, тыс. руб.'] == df['Цена, тыс. руб.'].max()])

In [ ]:
# Новое распределение цены
df['Цена, тыс. руб.'].plot(kind='hist',
        color='green', edgecolor='black', alpha=0.6,
        figsize=(10, 6))

plt.title('Распределение цены', size=18)
plt.xlabel('Цена (тыс. руб.)', size=14)
plt.ylabel('Частота', size=14)
plt.show()

In [ ]:
# Новое распределение стоимости по общей площади
ax1 = df[df['Тип дома'] == 'блочный'].plot(kind='scatter',
              x='Общая', y='Цена, тыс. руб.',
              color = 'orange', alpha=0.6,
              figsize=(10, 7))

df[df['Тип дома'] == 'каркасный'].plot(kind='scatter',
              x='Общая', y='Цена, тыс. руб.',
              color = 'blue', alpha=0.6,
              figsize=(10, 7), ax=ax1)

df[df['Тип дома'] == 'кирпичный'].plot(kind='scatter',
              x='Общая', y='Цена, тыс. руб.',
              color = 'red', alpha=0.6,
              figsize=(10, 7), ax=ax1)

plt.legend(labels=['Блочный дом', 'Каркасный дом', 'Кирпичный дом'])
plt.show()

Посмотрим, как стоимость распределяется в зависимости от других числовых параметров: таких, как число комнат, жилая площадь, площадь кухни.

In [ ]:
numeric_col = df.select_dtypes(include=['int', 'float64']).columns
df[numeric_col].corr()

Согласно матрице корреляции Пирсона, мы можем использовать число комнат, общую площадь, жилую площадь и площадь кухни в качестве независимых переменных. Но между общей и жилой площадями сильная линейная связь, то есть присутствует высокий риск мультиколлинеарности. Поэтому следует удалить один из них - это будет столбец "Жилая".

In [ ]:
df.pop('Жилая')
df

Определим все категориальные данные и преобразуем их с помощью класса OneHotEncoder, импортированного из библиотеки sklearn.

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns

encoder = OneHotEncoder()

encoded_categorical = encoder.fit_transform(df[categorical_cols])
header = encoder.get_feature_names_out(categorical_cols)
df_encoded = pd.DataFrame(encoded_categorical.toarray(), columns=header)

df_mlr = pd.concat([df.drop(categorical_cols, axis=1), df_encoded], axis=1)
df_mlr

Разделим данные на наборы для обучения и тестирования

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_mlr.drop('Цена, тыс. руб.', axis=1), df_mlr['Цена, тыс. руб.'], test_size=0.25, random_state=42)

Теперь мы можем построить модель.

Вызовем регрессор LinearRegression из библиотеки sklearn и передадим туда данные.

In [ ]:
mlr = LinearRegression()

mlr.fit(X_train, y_train)

print(f'a: {mlr.intercept_}')
print(f'b0 - b30: {mlr.coef_}')

Теперь, после создания модели мы можем спрогнозировать стоимость недвижимости по независимым переменным из тестового набора. 

In [ ]:
y_pred = mlr.predict(X_test)

Получив с помощью модели теоретические значения, мы можем сравнить их с действительными, которые содержатся в тестовом наборе.

Сначала оценим модель с помощью метода коэффициента детерминации, который показывает в долях, на сколько хорошо регрессионная модель соответствует реальным данным. Чем выше показатель, тем лучше.

In [ ]:
r2 = r2_score(y_test, y_pred)

print(f"R2: {r2:.4f}")

Оценка показала значение в 0.6836, что является типичным для моделей в области экономики. Но в целом для линейной регрессии такое значение является средним.

Дополнительно оценим модель с помощью других методов. Например,
* Метод средней абсолютной ошибки: измеряет среднее абсолютное отклонение между фактическими и прогнозируемыми значениями в наборе данных.
* Метод среднеквадратичной ошибки: показывает, насколько в среднем предсказания модели отклоняются от реальных значений.
* Корень из среднеквадратичной ошибки: Показывает среднюю величину ошибки

In [ ]:
# Метод средней абсолютной ошибки 
mae = mean_absolute_error(y_test, y_pred)
print(f'MAE: {mae:.2f}')

# Метод среднеквадратичной ошибки
mse = mean_squared_error(y_test, y_pred)
print(f'MSE: {mse:.2f}')

# Корень из среднеквадратичной ошибки
rmse = np.sqrt(mse)
print(f'RMSE: {rmse:.2f}')

В целом, модель получилась рабочей. Она способна объяснить большую часть зависимостей в данных. Модель определяет стоимость недвижимости с ошибкой в 353,9 тыс. руб, что может стать критичным в некоторых моментах. По оценке RMSE можно заметить, что в модели присутствуют выбросы, которые можно увидеть, исходя из разности между RMSE и MAE в 168,92 тыс. руб. Что касается улучшения, то в модель необходимо добавить параметры, по которым можно будет уточнить, почему стоимость недвижимости больше или меньше теоретического.